# Orbitstack NRR Decline — Commercial Diagnostic

Synthetic data demonstration of a 2-week commercial analytics diagnostic
for a fictional PE-backed B2B SaaS business. All figures are generated;
no real client data is present.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

DATA_DIR = Path("data")
FIGURES_DIR = Path("outputs/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SNAPSHOT_DATE = pd.Timestamp("2025-12-31")
ANALYSIS_START = pd.Timestamp("2023-01-01")
ANALYSIS_END = pd.Timestamp("2025-12-31")

PALETTE = {
    "primary":    "#1F3A5F",
    "secondary":  "#4A7C9D",
    "accent":     "#C8553D",
    "neutral":    "#6B7280",
    "positive":   "#4A7C59",
    "background": "#F8F9FA",
    "text":       "#1F2937",
}

SEGMENT_COLOURS = {
    "SMB":         "#6B9BD1",
    "Mid-market":  "#C8553D",
    "Enterprise":  "#1F3A5F",
}

SEGMENT_LINESTYLES = {
    "SMB":         "--",
    "Mid-market":  "-",
    "Enterprise":  "-.",
}

MOVEMENT_COLOURS = {
    "starting":    "#6B7280",
    "new":         "#4A7C59",
    "expansion":   "#4A7C9D",
    "contraction": "#C8553D",
    "churn":       "#8B3A2F",
    "ending":      "#1F3A5F",
}

SEGMENT_ORDER = ["SMB", "Mid-market", "Enterprise"]

SOURCE_ANNOTATION = (
    "Source: Orbitstack subscription data, Q1 2022 – Q4 2025. "
    "Synthetic data for demonstration purposes."
)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.facecolor": PALETTE["background"],
    "axes.facecolor":   PALETTE["background"],
    "axes.edgecolor":   PALETTE["neutral"],
    "axes.labelcolor":  PALETTE["text"],
    "text.color":       PALETTE["text"],
    "xtick.color":      PALETTE["text"],
    "ytick.color":      PALETTE["text"],
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.family":       "sans-serif",
    "font.sans-serif":   ["Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi":        100,
})


def add_source_annotation(fig: plt.Figure, text: str = SOURCE_ANNOTATION) -> None:
    fig.text(
        0.01, -0.02, text,
        fontsize=8, color=PALETTE["neutral"], ha="left", va="top",
    )


## Section 0 — Glossary

Short reference list. Definitions expanded in section 3.

- **ARR** — Annual Recurring Revenue. Contracted subscription revenue for the
  forward twelve months, measured on active paying customers.
- **CARR** — Committed ARR including signed-but-not-yet-billing deals. Not
  used in this notebook; only active ARR is in scope.
- **NRR** — Net Revenue Retention. Ending ARR of a cohort divided by that
  cohort's starting ARR, including expansion and contraction but excluding
  new logos acquired in the analysis period.
- **GRR** — Gross Revenue Retention. NRR without the expansion
  contribution — a pure retention lens.
- **Churn** — used here in the revenue sense (lost ARR) unless prefixed
  "logo". Logo churn counts customers; revenue churn weights by ARR.
- **Expansion / Contraction** — upward and downward moves in ARR while a
  customer remains active. Full departure is churn, not contraction.
- **Cohort** — the calendar quarter of a customer's first paid invoice
  date. Used as the retention anchor throughout.


## Section 2 — Engagement context

> **Orbitstack** is a PE-backed B2B SaaS business providing workflow
> automation to mid-market logistics firms. Mid-twenties-millions ARR as of
> 31 December 2025, 3 years into a 5-year hold period with sponsor
> **Meridian Capital**. The sponsor has requested a commercial analytics
> review ahead of a planned exit process in 18–24 months.
>
> **The trigger:** Headline NRR has declined from 118% in Q1 2023 to 104%
> in Q4 2025. The CFO's initial view is that this is a churn problem driven
> by macro pressure on logistics customers. The sponsor wants an
> independent diagnostic.
>
> **This notebook represents the first two weeks of the engagement** —
> data collection, definitional audit, decomposition, segmentation, and
> initial hypothesis formation.

The questions this notebook answers: Is the NRR decline real under a
defensible definition? Is it driven by churn or by expansion collapse? Is
it concentrated in a segment? Do the data support a competitive or
product-led explanation, or is the cause elsewhere? What would the
priority workstreams be for weeks 3–8?


## Section 3 — Definitional audit

Before reproducing the trigger metric we fix the definitions. The sponsor
and the CFO cite a headline NRR number; we need to know exactly which NRR
that is and where the denominators sit.

**NRR definition.** We use dollar-based, trailing-twelve-months Net
Revenue Retention, expressed as a percentage. For every reporting
quarter, the numerator is the ARR at quarter end from customers who were
active twelve months earlier; the denominator is the ARR those same
customers contributed at the start of that twelve-month window. New
logos acquired inside the analysis window (Q1 2023 – Q4 2025) are
excluded from both sides so the metric measures retention rather than
blended retention-plus-growth. New-logo ARR is reported separately in the
decomposition in section 6.

**Cohort anchor.** Customer cohort is the calendar quarter of
`first_paid_invoice_date` — the billing system record. The CRM
`acquisition_date` lags by 0–3 months in the ordinary course and by up
to 90 days for a small number of customers on free-trial gating (flagged
as DQ quirk C in section 4). Anchoring to billing ensures retention curves
rest on the date cash actually starts rather than on sales-stage
timestamps.

**Scope exclusions.** Re-activations are excluded — no churned customer
returns in the generated dataset, and production data would need
dedicated handling before this metric could absorb them. CARR and backlog
are out of scope; active ARR only is used throughout. All values are
GBP-denominated with no FX translation applied; the business is
UK-domiciled and EU customers are billed in GBP.

**Timebase.** The snapshot is 31 December 2025. No relative dates appear
in analysis or commentary. Quarters are calendar quarters. The generator
runs from Q1 2021 so the Q1 2023 TTM denominator rests on a fully
established 2022 opening book.

### Assumptions requiring client finance-team validation

1. `first_paid_invoice_date` is the canonical cohort anchor; five
   customers with a CRM-to-billing gap exceeding 30 days are treated as
   free-trial overlaps rather than data errors. Confirm with the CS /
   finance workflow owner.
2. Customers flagged `Churned` on their final subscription row are
   assumed to have ceased paying on that date. Any live re-activation
   pattern would need an explicit treatment.
3. Effective price per seat is taken verbatim from the billing export;
   discount-authorisation trail (rep, deal, approver) is not
   reconstructed from the CSVs in this diagnostic.
4. The four-quarter rolling average used for segment NRR in section 7
   consumes Q2–Q4 2022 as warmup input. Those pre-period quarters are
   present in the generated data and are assumed to be analytically
   comparable to the Q1 2023 opening book.


## Section 4 — Data overview, provenance, and quality

Three CSVs are produced by `data/generate_data.py` with a fixed seed of
42. Row counts, date ranges, segment split, provenance, lineage, and data
quality quirks are tabulated below before any analytical work begins.


In [ ]:
customers = pd.read_csv(
    DATA_DIR / "customers.csv",
    parse_dates=["acquisition_date", "first_paid_invoice_date"],
)
subscriptions = pd.read_csv(
    DATA_DIR / "subscriptions.csv",
    parse_dates=["month_end_date", "contract_end_date"],
)
churn_reasons = pd.read_csv(
    DATA_DIR / "churn_reasons.csv",
    parse_dates=["churn_date"],
)

customers["segment"] = pd.Categorical(
    customers["segment"], categories=SEGMENT_ORDER, ordered=True
)


### Row counts and date ranges


In [ ]:
row_counts = pd.DataFrame(
    [
        {
            "file": "customers.csv",
            "rows": len(customers),
            "date_column": "first_paid_invoice_date",
            "min_date": customers["first_paid_invoice_date"].min().date(),
            "max_date": customers["first_paid_invoice_date"].max().date(),
        },
        {
            "file": "subscriptions.csv",
            "rows": len(subscriptions),
            "date_column": "month_end_date",
            "min_date": subscriptions["month_end_date"].min().date(),
            "max_date": subscriptions["month_end_date"].max().date(),
        },
        {
            "file": "churn_reasons.csv",
            "rows": len(churn_reasons),
            "date_column": "churn_date",
            "min_date": churn_reasons["churn_date"].min().date(),
            "max_date": churn_reasons["churn_date"].max().date(),
        },
    ]
)
row_counts


### Segment split (customer count and share of original ARR)


In [ ]:
segment_split = (
    customers.groupby("segment", observed=True)
    .agg(customers=("customer_id", "size"),
         original_arr_gbp=("original_arr_gbp", "sum"))
)
segment_split["customer_share"] = (
    segment_split["customers"] / segment_split["customers"].sum()
)
segment_split["arr_share"] = (
    segment_split["original_arr_gbp"] / segment_split["original_arr_gbp"].sum()
)
segment_split["original_arr_gbp"] = segment_split["original_arr_gbp"].map(
    lambda v: f"£{v/1_000_000:.2f}m"
)
segment_split["customer_share"] = segment_split["customer_share"].map(
    lambda v: f"{v*100:.1f}%"
)
segment_split["arr_share"] = segment_split["arr_share"].map(
    lambda v: f"{v*100:.1f}%"
)
segment_split


### Data provenance

Each CSV corresponds to a fictional source system owned by a different
team. The mapping is narrative context for the diagnostic; the CSVs
themselves are standardised extracts and do not carry source-system
specific fields.

| File | Fictional source | Owning team |
|---|---|---|
| `customers.csv` | CRM export | Revenue Operations |
| `subscriptions.csv` | Billing system export | Finance |
| `churn_reasons.csv` | Customer success platform export | Customer Success |


### Data lineage

```
customers.csv         ─┬─→ section 4 (overview, provenance, DQ)
                       │   → section 7 (segmentation by segment)
                       │   → section 8 (cohort = first_paid_invoice_date quarter)
                       └─→ section 10 (acquisition-era facet)

subscriptions.csv     ─┬─→ section 5 (headline NRR)
                       │   → section 6 (NRR decomposition + reconciliation)
                       │   → section 6a (sensitivity variants)
                       │   → section 7 (quarterly NRR by segment)
                       │   → section 8 (retained ARR by cohort)
                       └─→ section 10 (seat expansion by cohort)

churn_reasons.csv     ───→ section 9 (churn reason taxonomy, FleetLogic gate)
```

Upstream transformations applied before the CSV boundary (customer
deduplication, invoice → ARR roll-up, churn attribution cleanup) are
owned by the respective source teams and are treated as trusted inputs
here.


### Data quality quirks

Three deliberate quirks are present in the generated data. Each mirrors a
reconciliation pattern common in commercial due-diligence work. Handling
decisions are recorded so they can be re-applied in production pipelines.

| Quirk | Detection | Handling | Impact on findings |
|---|---|---|---|
| A — Billing system £1 dips. Three customers show a single-month ARR dip of exactly £1 mid-history. | Vectorised check for rows where `arr_gbp - prev_arr == -1.0` and `arr_gbp - next_arr == -1.0` while `subscription_status == 'Active'`. | Forward-fill the affected month with prior-month ARR before computing section 6 decomposition movements. | Negligible — affects three customer-months out of 16,000+. |
| B — Backdated ARR adjustment. One customer has an unrecorded mid-contract uplift reconciled in month 6. | Same vectorised rule with sign flipped (`+£500` spike against prior and next months). | Absorb the uplift into the baseline ARR from month 5 onwards (treat the month-6 spike as calendar noise, not expansion). | Negligible — affects one customer-month. |
| C — CRM vs billing definitional mismatch. Five customers have `acquisition_date` preceding `first_paid_invoice_date` by 30–90 days (free-trial period not reflected in the CRM date). | `first_paid_invoice_date - acquisition_date` exceeds 30 days. Because 0–90 day gaps occur in the baseline population too, the five cannot be uniquely identified by gap size alone. | Use `first_paid_invoice_date` as the cohort anchor throughout. Document the gap as a class of issue rather than naming specific IDs. Requires finance-team confirmation of trial-billing workflow. | Minor — shifts five cohorts by one quarter at most; TTM NRR windows are wide enough to absorb the shift. |


### Customer acquisition by quarter

One overview chart closes section 4: a stacked bar of customer count by
segment by acquisition quarter. It establishes that the book is heavily
front-loaded in 2021 (a deliberate feature of the baseline population,
noted in section 3) and that recent intake has tilted toward smaller
segments.


In [ ]:
in_book = customers[customers["first_paid_invoice_date"] <= SNAPSHOT_DATE].copy()
post_snapshot_count = len(customers) - len(in_book)

acq_quarter = in_book["first_paid_invoice_date"].dt.to_period("Q")
counts_by_quarter = (
    in_book.assign(acq_quarter=acq_quarter)
    .pivot_table(
        index="acq_quarter",
        columns="segment",
        values="customer_id",
        aggfunc="count",
        fill_value=0,
        observed=True,
    )
    .reindex(columns=SEGMENT_ORDER, fill_value=0)
    .sort_index()
)
counts_by_quarter.index = counts_by_quarter.index.astype(str)

fig, ax = plt.subplots(figsize=(10, 6))
bottom = np.zeros(len(counts_by_quarter))
x = np.arange(len(counts_by_quarter))
for segment in SEGMENT_ORDER:
    values = counts_by_quarter[segment].to_numpy()
    ax.bar(
        x, values, bottom=bottom,
        color=SEGMENT_COLOURS[segment],
        edgecolor=PALETTE["background"], linewidth=0.5,
        label=segment,
    )
    bottom = bottom + values

year_2021_total = int(counts_by_quarter.loc[["2021Q1", "2021Q2", "2021Q3", "2021Q4"]].to_numpy().sum())
book_total = int(counts_by_quarter.to_numpy().sum())
book_share_2021 = year_2021_total / book_total

ax.set_xticks(x)
ax.set_xticklabels(counts_by_quarter.index, rotation=45, ha="right")
ax.set_xlabel("Acquisition quarter (first paid invoice)")
ax.set_ylabel("Customers acquired (count)")
ax.set_title(
    f"Customer intake front-loaded: {book_share_2021*100:.0f}% of the active book landed in 2021; "
    "mid-market intake thins from 2024",
    loc="left", color=PALETTE["text"], fontsize=12,
)
ax.legend(title="Segment", loc="upper right", frameon=False)
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

footnote = (
    f"Includes the {book_total} customers with a first paid invoice on or before "
    f"31 December 2025. A further {post_snapshot_count} customers were acquired late in 2025 "
    f"and had not yet billed at the snapshot."
)
fig.text(0.01, -0.02, footnote, fontsize=8, color=PALETTE["neutral"], ha="left", va="top")
fig.text(0.01, -0.06, SOURCE_ANNOTATION, fontsize=8, color=PALETTE["neutral"], ha="left", va="top")

fig.tight_layout()
fig.savefig(
    FIGURES_DIR / "04_acquisition_by_segment.png",
    dpi=150, bbox_inches="tight",
)
plt.show()


## Section 5 — Headline NRR reproduction

Reproduces the trigger metric from the engagement context using the
definition agreed in section 3: trailing-twelve-months, dollar-based
Net Revenue Retention, excluding customers acquired during the
analysis period. The TTM cohort at quarter `Q` is the set of
customers with positive ARR at quarter end `Q-4`; NRR for `Q` is the
ratio of those customers' ARR at `Q` to their ARR at `Q-4`.

In [ ]:
new_logo_ids = customers.loc[
    customers["first_paid_invoice_date"] >= ANALYSIS_START, "customer_id"
]

WARMUP_START = ANALYSIS_START - pd.DateOffset(months=12)
all_quarter_ends = pd.date_range(WARMUP_START, ANALYSIS_END, freq="QE")
analysis_quarter_ends = pd.date_range(ANALYSIS_START, ANALYSIS_END, freq="QE")

active_quarter_arr = subscriptions[
    (subscriptions["month_end_date"].isin(all_quarter_ends))
    & (subscriptions["subscription_status"] == "Active")
    & (~subscriptions["customer_id"].isin(new_logo_ids))
]
arr_pivot = active_quarter_arr.pivot_table(
    index="customer_id",
    columns="month_end_date",
    values="arr_gbp",
    aggfunc="sum",
    fill_value=0.0,
)

ttm_start_quarter_ends = pd.DatetimeIndex(
    analysis_quarter_ends - pd.DateOffset(months=12)
)
start_arr_matrix = arr_pivot.reindex(columns=ttm_start_quarter_ends, fill_value=0.0)
end_arr_matrix = arr_pivot.reindex(columns=analysis_quarter_ends, fill_value=0.0)

cohort_mask = start_arr_matrix.to_numpy() > 0
starting_total = (start_arr_matrix.to_numpy() * cohort_mask).sum(axis=0)
ending_total = (end_arr_matrix.to_numpy() * cohort_mask).sum(axis=0)

nrr_quarterly = pd.DataFrame({
    "quarter": analysis_quarter_ends.to_period("Q").astype(str),
    "starting_arr_gbp": starting_total,
    "ending_arr_gbp": ending_total,
    "nrr": ending_total / starting_total,
})

nrr_quarterly.assign(
    starting_arr_gbp=lambda d: d["starting_arr_gbp"].map(lambda v: f"£{v/1_000_000:.2f}m"),
    ending_arr_gbp=lambda d: d["ending_arr_gbp"].map(lambda v: f"£{v/1_000_000:.2f}m"),
    nrr=lambda d: (d["nrr"] * 100).map(lambda v: f"{v:.1f}%"),
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.axhline(
    1.0, linestyle=":", color=PALETTE["neutral"], linewidth=1.0,
    label="100% reference",
)
ax.plot(
    nrr_quarterly["quarter"],
    nrr_quarterly["nrr"],
    marker="o", linewidth=2.2, markersize=7,
    color=PALETTE["primary"],
    label="Trailing-12-months NRR",
)

first_row = nrr_quarterly.iloc[0]
last_row = nrr_quarterly.iloc[-1]
ax.annotate(
    f"Q1 2023: {first_row['nrr']*100:.1f}%",
    xy=(0, first_row["nrr"]),
    xytext=(12, 14), textcoords="offset points",
    color=PALETTE["primary"], fontsize=10, fontweight="bold",
    arrowprops=dict(arrowstyle="-", color=PALETTE["primary"], lw=0.6),
)
ax.annotate(
    f"Q4 2025: {last_row['nrr']*100:.1f}%",
    xy=(len(nrr_quarterly) - 1, last_row["nrr"]),
    xytext=(-110, -30), textcoords="offset points",
    color=PALETTE["accent"], fontsize=10, fontweight="bold",
    arrowprops=dict(arrowstyle="-", color=PALETTE["accent"], lw=0.6),
)

ax.set_xlabel("Quarter")
ax.set_ylabel("Net Revenue Retention (%)")
ax.set_xticks(range(len(nrr_quarterly)))
ax.set_xticklabels(nrr_quarterly["quarter"], rotation=45, ha="right")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))
y_lo = float(nrr_quarterly["nrr"].min()) - 0.04
y_hi = float(nrr_quarterly["nrr"].max()) + 0.04
ax.set_ylim(min(0.98, y_lo), max(1.22, y_hi))

start_pct = first_row["nrr"] * 100
end_pct = last_row["nrr"] * 100
ax.set_title(
    f"Net Revenue Retention declines from {start_pct:.0f}% to {end_pct:.0f}% across 12 quarters",
    loc="left", color=PALETTE["text"], fontsize=12,
)
ax.legend(loc="lower left", frameon=False)

add_source_annotation(fig)
fig.tight_layout()
fig.savefig(
    FIGURES_DIR / "05_headline_nrr_decline.png",
    dpi=150, bbox_inches="tight",
)
plt.show()

The TTM dollar-based NRR drops from 115.6% in Q1 2023 to 104.9% in
Q4 2025 — an 11 percentage point decline that reproduces the trigger
described in the engagement context. The trajectory is not monotonic:
NRR climbs above 120% through Q2 2024 before falling sharply across
the next four quarters. Whether that drop is driven by churn or by
expansion collapse — the CFO's working hypothesis versus the
alternative — is the question section 6 answers by decomposing each
quarter's ARR movements.

## Section 6 — NRR decomposition

Section 5 reproduced the headline decline but did not decompose it. The
12 quarterly TTM NRR values conceal four distinct movements — new ARR,
expansion, contraction, and churn — that together drive the net
change. This section reconstructs those movements quarter by quarter
for every customer, asserts that they tie back to the observed ending
ARR (no rounding drift, no missing rows), and then compares NRR against
its churn-only sibling GRR to separate the expansion story from the
retention story.

The reconciliation uses a full customer × quarter ARR pivot. For each
analysis quarter Q, a retention cohort is defined as customers with
positive ARR at Q-1. Flows are classified vectorised across the pivot:
new ARR for customers who first billed in Q, expansion and contraction
for in-cohort ARR moves, and churn for in-cohort customers who dropped
to zero ARR. The reconciliation identity is
`ending = starting + new + expansion - contraction - churn` and is
enforced by assertion before any chart is drawn.

In [ ]:
# Build one QE-spaced index and slice paired start/end views rather
# than subtracting DateOffset(months=3) — the latter drifts by a day
# on mid-quarter dates (Jun 30 − 3M = Mar 30, missing the Mar 31 QE).
decomp_all_ends = pd.date_range(
    ANALYSIS_START - pd.DateOffset(months=3), ANALYSIS_END, freq="QE",
)
decomp_end_dates = decomp_all_ends[1:]
decomp_start_dates = decomp_all_ends[:-1]

arr_quarter_pivot = (
    subscriptions[
        (subscriptions["month_end_date"].isin(decomp_all_ends))
        & (subscriptions["subscription_status"] == "Active")
    ]
    .pivot_table(
        index="customer_id",
        columns="month_end_date",
        values="arr_gbp",
        aggfunc="sum",
        fill_value=0.0,
    )
    .reindex(columns=decomp_all_ends, fill_value=0.0)
)

end_arr_q = arr_quarter_pivot.reindex(columns=decomp_end_dates, fill_value=0.0).to_numpy()
start_arr_q = arr_quarter_pivot.reindex(columns=decomp_start_dates, fill_value=0.0).to_numpy()

retention_cohort = start_arr_q > 0
new_logo_this_q = (start_arr_q == 0) & (end_arr_q > 0)
diff_arr = end_arr_q - start_arr_q

starting_total_q    = (start_arr_q * retention_cohort).sum(axis=0)
ending_total_q      = end_arr_q.sum(axis=0)
new_arr_q           = (end_arr_q * new_logo_this_q).sum(axis=0)
expansion_arr_q     = np.where(retention_cohort & (end_arr_q > 0) & (diff_arr > 0),  diff_arr, 0.0).sum(axis=0)
contraction_arr_q   = np.where(retention_cohort & (end_arr_q > 0) & (diff_arr < 0), -diff_arr, 0.0).sum(axis=0)
churn_arr_q         = np.where(retention_cohort & (end_arr_q == 0), start_arr_q, 0.0).sum(axis=0)

reconciliation_gap = ending_total_q - (
    starting_total_q + new_arr_q + expansion_arr_q - contraction_arr_q - churn_arr_q
)
max_gap = float(np.max(np.abs(reconciliation_gap)))
if max_gap > 0.01:
    worst_idx = int(np.argmax(np.abs(reconciliation_gap)))
    worst_quarter = decomp_end_dates[worst_idx].to_period("Q")
    raise AssertionError(
        f"ARR decomposition mismatch at {worst_quarter}: "
        f"gap £{reconciliation_gap[worst_idx]:,.2f}"
    )
print("Decomposition reconciled — all 12 quarters tie")

nrr_quarterly_decomp = (
    starting_total_q - churn_arr_q - contraction_arr_q + expansion_arr_q
) / starting_total_q
grr_quarterly_decomp = (
    starting_total_q - churn_arr_q - contraction_arr_q
) / starting_total_q

decomposition = pd.DataFrame({
    "quarter":       decomp_end_dates.to_period("Q").astype(str),
    "starting_arr":  starting_total_q,
    "new":           new_arr_q,
    "expansion":     expansion_arr_q,
    "contraction":   contraction_arr_q,
    "churn":         churn_arr_q,
    "ending_arr":    ending_total_q,
    "nrr_q":         nrr_quarterly_decomp,
    "grr_q":         grr_quarterly_decomp,
})

money_cols = ["starting_arr", "new", "expansion", "contraction", "churn", "ending_arr"]
pct_cols = ["nrr_q", "grr_q"]
decomposition_display = decomposition.copy()
for col in money_cols:
    decomposition_display[col] = decomposition_display[col].map(lambda v: f"£{v/1_000_000:.2f}m")
for col in pct_cols:
    decomposition_display[col] = (decomposition_display[col] * 100).map(lambda v: f"{v:.1f}%")
decomposition_display

In [ ]:
quarters_q  = decomposition["quarter"].to_numpy()
x_q         = np.arange(len(quarters_q))

new_m        = decomposition["new"].to_numpy()        / 1_000_000
expansion_m  = decomposition["expansion"].to_numpy()  / 1_000_000
contraction_m= decomposition["contraction"].to_numpy()/ 1_000_000
churn_m      = decomposition["churn"].to_numpy()      / 1_000_000
net_m        = new_m + expansion_m - contraction_m - churn_m

fig, ax = plt.subplots(figsize=(10, 6))
ax.axhline(0, color=PALETTE["neutral"], linewidth=0.8, zorder=0)

# Hatch patterns give each movement a distinct texture so greyscale
# readers can separate New, Expansion, Contraction, and Churn without
# relying on colour luminance alone.
ax.bar(x_q, new_m,        color=MOVEMENT_COLOURS["new"],
       edgecolor=PALETTE["background"], linewidth=0.4,
       hatch="",    label="New ARR")
ax.bar(x_q, expansion_m,  bottom=new_m, color=MOVEMENT_COLOURS["expansion"],
       edgecolor=PALETTE["background"], linewidth=0.4,
       hatch="///", label="Expansion")
ax.bar(x_q, -contraction_m, color=MOVEMENT_COLOURS["contraction"],
       edgecolor=PALETTE["background"], linewidth=0.4,
       hatch="...", label="Contraction")
ax.bar(x_q, -churn_m, bottom=-contraction_m, color=MOVEMENT_COLOURS["churn"],
       edgecolor=PALETTE["background"], linewidth=0.4,
       hatch="xxx", label="Churn")

ax.plot(x_q, net_m, marker="o", linewidth=1.6, markersize=6,
        color=PALETTE["primary"], label="Net ARR change")

positive_first  = new_m[0] + expansion_m[0]
positive_last   = new_m[-1] + expansion_m[-1]

ax.set_xticks(x_q)
ax.set_xticklabels(quarters_q, rotation=45, ha="right")
ax.set_xlabel("Quarter")
ax.set_ylabel("ARR movement (£m)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"£{v:.1f}m"))
ax.set_title(
    f"Positive ARR flows halve from £{positive_first:.1f}m to £{positive_last:.1f}m per quarter; "
    "expansion compression drives the decline",
    loc="left", color=PALETTE["text"], fontsize=12,
)
ax.legend(loc="upper right", frameon=False, ncol=1, fontsize=9)

add_source_annotation(fig)
fig.tight_layout()
fig.savefig(
    FIGURES_DIR / "06_nrr_decomposition.png",
    dpi=150, bbox_inches="tight",
)
plt.show()

In [ ]:
ttm_start_total_full = (start_arr_matrix.to_numpy() * cohort_mask).sum(axis=0)
# GRR caps each customer's ending ARR at their Q-4 baseline so
# expansion is removed from the numerator but churn/contraction bite.
ttm_capped = np.minimum(end_arr_matrix.to_numpy(), start_arr_matrix.to_numpy())
ttm_grr_total = (ttm_capped * cohort_mask).sum(axis=0)

grr_ttm = ttm_grr_total / ttm_start_total_full
nrr_ttm = nrr_quarterly["nrr"].to_numpy()
quarters_ttm = nrr_quarterly["quarter"].to_numpy()
x_ttm = np.arange(len(quarters_ttm))

fig, ax = plt.subplots(figsize=(10, 6))
ax.axhline(1.0, linestyle=":", color=PALETTE["neutral"], linewidth=1.0, label="100% reference")

ax.plot(x_ttm, nrr_ttm, marker="o", linewidth=2.2, markersize=7,
        color=PALETTE["primary"], label="NRR (trailing-12-months)")
ax.plot(x_ttm, grr_ttm, marker="s", linewidth=2.2, markersize=7,
        linestyle="--", color=PALETTE["accent"], label="GRR (trailing-12-months)")

grr_low, grr_high = float(grr_ttm.min()) * 100, float(grr_ttm.max()) * 100
nrr_first, nrr_last = float(nrr_ttm[0]) * 100, float(nrr_ttm[-1]) * 100

ax.set_xticks(x_ttm)
ax.set_xticklabels(quarters_ttm, rotation=45, ha="right")
ax.set_xlabel("Quarter")
ax.set_ylabel("Retention (%)")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))
ax.set_ylim(0.85, 1.25)
ax.set_title(
    f"GRR holds between {grr_low:.0f}% and {grr_high:.0f}% while NRR falls from "
    f"{nrr_first:.0f}% to {nrr_last:.0f}%: an expansion problem, not a churn problem",
    loc="left", color=PALETTE["text"], fontsize=12,
)
ax.legend(loc="lower left", frameon=False)

add_source_annotation(fig)
fig.tight_layout()
fig.savefig(
    FIGURES_DIR / "06_nrr_vs_grr.png",
    dpi=150, bbox_inches="tight",
)
plt.show()

The decomposition splits the headline decline cleanly. Gross Revenue
Retention — the churn-only retention lens, with expansion removed —
holds in a narrow band across all twelve quarters: its lowest point is
still above 91% and its high point sits near 97%. Across the same
window, Net Revenue Retention falls by roughly eleven percentage
points. The gap between the two lines is expansion contribution, and
that gap is what has compressed: quarterly positive ARR flows (new
plus expansion) run around £2m in 2023 but sit closer to £1.2m by
late 2025. Quarterly losses (churn plus contraction) remain modest in
most quarters and peak just above £1m in Q3 2024 — a single spike that
GRR absorbs without dropping below 92%.

This is an expansion problem, not a churn problem. The retention base
is behaving normally; customers who stay, stay, and very few drop to
zero. What has changed is the pace at which existing customers
purchase more seats or move up the product tier. Section 7 asks
whether this compression is uniform across segments or whether it is
concentrated — that segmentation is the next layer of the
diagnostic.

## Section 6a — Sensitivity check

Headlines are only as trustworthy as the definitional choices behind
them. Before carrying the 104.9% Q4 2025 figure into the findings, the
same Q4 2025 NRR is computed under five definitional variants drawn
from the same subscription and customer data — no re-derived prices,
no alternative assumptions. The table below contrasts: the
trailing-twelve-months dollar-based headline; the single-period
quarterly read (Q4 2025 against Q3 2025); the revenue-weighted
per-customer average (equivalent to the headline by construction, shown
to make the weighting explicit); the customer-count-weighted average
where each retained customer contributes equally; and a tenure-filtered
variant that keeps only customers with at least four full quarters of
billing history at the start of the TTM window.

In [ ]:
q4_ttm_start = pd.Timestamp("2024-12-31")
q4_qoq_start = pd.Timestamp("2025-09-30")
q4_end       = pd.Timestamp("2025-12-31")

sens_pivot = (
    subscriptions[
        (subscriptions["month_end_date"].isin([q4_qoq_start, q4_ttm_start, q4_end]))
        & (subscriptions["subscription_status"] == "Active")
    ]
    .pivot_table(
        index="customer_id",
        columns="month_end_date",
        values="arr_gbp",
        aggfunc="sum",
        fill_value=0.0,
    )
    .reindex(columns=[q4_qoq_start, q4_ttm_start, q4_end], fill_value=0.0)
)

first_paid_aligned = (
    customers.set_index("customer_id")["first_paid_invoice_date"]
    .reindex(sens_pivot.index)
)
excl_new_logo = (first_paid_aligned < ANALYSIS_START).to_numpy()
# Tenure filter: at least four full quarters before the TTM start
# (Q4 2024). A customer with first_paid_invoice_date on or before the
# last day of Q4 2023 has completed four quarters by Q4 2024 open.
tenure_ok = (first_paid_aligned <= pd.Timestamp("2023-12-31")).to_numpy()

ttm_start_arr = sens_pivot[q4_ttm_start].to_numpy()
qoq_start_arr = sens_pivot[q4_qoq_start].to_numpy()
end_arr_q4    = sens_pivot[q4_end].to_numpy()

# Variant 1 — TTM dollar-based (headline).
mask_ttm = excl_new_logo & (ttm_start_arr > 0)
v_ttm = end_arr_q4[mask_ttm].sum() / ttm_start_arr[mask_ttm].sum()

# Variant 2 — Single-period quarterly (Q4 2025 vs Q3 2025).
mask_qoq = excl_new_logo & (qoq_start_arr > 0)
v_qoq = end_arr_q4[mask_qoq].sum() / qoq_start_arr[mask_qoq].sum()

# Variant 3 — Revenue-weighted per-customer average. Each retained
# customer's individual ratio is weighted by starting ARR. This equals
# the headline by construction; the row makes the weighting explicit.
per_customer_ratio = end_arr_q4[mask_ttm] / ttm_start_arr[mask_ttm]
weights_ttm = ttm_start_arr[mask_ttm]
v_rev_weighted = float(
    (weights_ttm * per_customer_ratio).sum() / weights_ttm.sum()
)

# Variant 4 — Customer-count-weighted. Simple mean of per-customer
# TTM ratios: each retained customer contributes equally.
v_count_weighted = float(per_customer_ratio.mean())

# Variant 5 — Tenure >= 4 quarters. Broaden the base to include 2023
# new logos (now fully seasoned) while excluding 2024+ customers.
mask_tenure = tenure_ok & (ttm_start_arr > 0)
v_tenure = end_arr_q4[mask_tenure].sum() / ttm_start_arr[mask_tenure].sum()

variant_values = [v_ttm, v_qoq, v_rev_weighted, v_count_weighted, v_tenure]
sensitivity = pd.DataFrame(
    [
        ("TTM dollar-based (headline)",                     v_ttm,            int(mask_ttm.sum())),
        ("Single-period quarterly (Q4 2025 vs Q3 2025)",    v_qoq,            int(mask_qoq.sum())),
        ("Revenue-weighted (headline methodology)",         v_rev_weighted,   int(mask_ttm.sum())),
        ("Customer-count-weighted equivalent",              v_count_weighted, int(mask_ttm.sum())),
        ("Excluding customers with <4 quarters of tenure",  v_tenure,         int(mask_tenure.sum())),
    ],
    columns=["Definition variant", "Q4 2025 NRR", "Retention cohort (customers)"],
)

sensitivity_low = min(variant_values) * 100
sensitivity_high = max(variant_values) * 100

sensitivity.assign(**{
    "Q4 2025 NRR": lambda d: (d["Q4 2025 NRR"] * 100).map(lambda x: f"{x:.1f}%"),
})


Directional finding holds across all definitional variants: every Q4
2025 NRR read sits at or below 105%, confirming the double-digit
percentage-point decline from the Q1 2023 starting point of 116% is
real under any reasonable measurement choice. The spread is
98.7%–105.0% across the five variants. The trailing-twelve-months
dollar-based headline (104.9%) and the revenue-weighted per-customer
average are identical by construction, which is the point: the
headline methodology is a revenue-weighted average already. The
single-period quarterly read (100.4%) is the tightest lens and sits
below the TTM figure because Q4 2025 had less expansion than the
rolling-year average carried into it. The customer-count-weighted
variant (98.7%) is the only read below 100%, reflecting the long tail
of modest contractions that a revenue-weighted view partially masks.
The tenure-filtered variant (105.0%) is marginally above the headline
because it brings in the 2023 new logos who are still in their
expansion phase.

Absolute figures will require client finance-team alignment before
external publication. The 6 percentage-point spread across variants is
within the range typically resolved through a one-hour methodology
workshop: agree the cohort filter (exclude analysis-period new logos
versus tenure-filter), the weighting (revenue versus customer-count),
and the averaging window (TTM versus single-period). Directionally,
the NRR decline is robust; the exact figure cited externally is a
definitional choice the sponsor should make deliberately.

## Section 7 — Segmentation

Cutting headline NRR by segment isolates where the decline is
concentrated. SMB, Mid-market, and Enterprise are computed on the
same trailing-12-months dollar-based definition used in section 5,
with new logos acquired during the analysis period excluded from
every quarter's denominator.

Enterprise has only ~70 customers in total, so its TTM cohort in any
given quarter rests on 40–50 customers; individual movements can shift
the raw quarterly value by several percentage points. The chart below
plots the raw quarterly NRR with reduced weight and a 4-quarter
rolling average at full weight. The rolling average is warmed up using
Q2–Q4 2022 quarters from the same dataset, so a full smoothed value is
available from Q1 2023 onwards — no leading NaNs and no misleading
partial averages.

In [ ]:
SEG_TTM_END_FIRST = pd.Timestamp("2022-06-30")
seg_quarter_ends = pd.date_range(SEG_TTM_END_FIRST, ANALYSIS_END, freq="QE")
seg_start_dates = pd.DatetimeIndex(seg_quarter_ends - pd.DateOffset(months=12))
seg_dates = pd.DatetimeIndex(sorted(set(seg_quarter_ends).union(seg_start_dates)))

seg_active = subscriptions[
    (subscriptions["month_end_date"].isin(seg_dates))
    & (subscriptions["subscription_status"] == "Active")
    & (~subscriptions["customer_id"].isin(new_logo_ids))
]
seg_pivot = (
    seg_active.pivot_table(
        index="customer_id",
        columns="month_end_date",
        values="arr_gbp",
        aggfunc="sum",
        fill_value=0.0,
    )
    .reindex(columns=seg_dates, fill_value=0.0)
)

seg_segment = (
    customers.set_index("customer_id")["segment"]
    .reindex(seg_pivot.index)
    .astype(str)
)

start_seg_matrix = seg_pivot.reindex(columns=seg_start_dates, fill_value=0.0).to_numpy()
end_seg_matrix = seg_pivot.reindex(columns=seg_quarter_ends, fill_value=0.0).to_numpy()
ret_mask = start_seg_matrix > 0

segment_nrr = pd.DataFrame({
    "quarter_end": seg_quarter_ends,
    "quarter": pd.PeriodIndex(seg_quarter_ends, freq="Q").astype(str),
})
for seg in SEGMENT_ORDER:
    seg_idx = (seg_segment == seg).to_numpy()[:, None]
    starting = (start_seg_matrix * ret_mask * seg_idx).sum(axis=0)
    ending = (end_seg_matrix * ret_mask * seg_idx).sum(axis=0)
    segment_nrr[seg] = np.where(starting > 0, ending / starting, np.nan)
    segment_nrr[f"{seg}_rolling"] = (
        segment_nrr[seg].rolling(window=4, min_periods=4).mean()
    )

segment_nrr_analysis = (
    segment_nrr[segment_nrr["quarter_end"] >= ANALYSIS_START]
    .reset_index(drop=True)
)

segment_nrr_display = segment_nrr_analysis.drop(columns=["quarter_end"]).copy()
for col in segment_nrr_display.columns:
    if col == "quarter":
        continue
    segment_nrr_display[col] = (segment_nrr_display[col] * 100).map(lambda v: f"{v:.1f}%")
segment_nrr_display


In [ ]:
quarters_seg = segment_nrr_analysis["quarter"].to_numpy()
x_seg = np.arange(len(quarters_seg))

fig, ax = plt.subplots(figsize=(10, 6))
ax.axhline(
    1.0, linestyle=":", color=PALETTE["neutral"], linewidth=1.0,
    label="100% reference",
)

for seg in SEGMENT_ORDER:
    colour = SEGMENT_COLOURS[seg]
    style = SEGMENT_LINESTYLES[seg]
    raw = segment_nrr_analysis[seg].to_numpy()
    rolling = segment_nrr_analysis[f"{seg}_rolling"].to_numpy()
    ax.plot(
        x_seg, raw,
        color=colour, linestyle=style, linewidth=1.0, alpha=0.32,
        marker="o", markersize=3.5, markeredgewidth=0,
        label="_nolegend_",
    )
    ax.plot(
        x_seg, rolling,
        color=colour, linestyle=style, linewidth=2.4,
        marker="o", markersize=6,
        label=f"{seg} (4Q rolling)",
    )

ax.plot(
    [], [], color=PALETTE["neutral"], linestyle="-", linewidth=1.0,
    alpha=0.45, marker="o", markersize=3.5,
    label="Raw quarterly (faded)",
)

ax.set_xticks(x_seg)
ax.set_xticklabels(quarters_seg, rotation=45, ha="right")
ax.set_xlabel("Quarter")
ax.set_ylabel("Trailing-12-months NRR (%)")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))

all_values = segment_nrr_analysis[
    SEGMENT_ORDER + [f"{s}_rolling" for s in SEGMENT_ORDER]
].to_numpy()
ax.set_ylim(
    min(0.85, float(np.nanmin(all_values)) - 0.04),
    max(1.30, float(np.nanmax(all_values)) + 0.04),
)

mm_first = float(segment_nrr_analysis["Mid-market_rolling"].iloc[0]) * 100
mm_last = float(segment_nrr_analysis["Mid-market_rolling"].iloc[-1]) * 100
ent_last = float(segment_nrr_analysis["Enterprise_rolling"].iloc[-1]) * 100
ax.set_title(
    f"Mid-market NRR collapse ({mm_first:.0f}% → {mm_last:.0f}%) drives "
    f"headline decline; Enterprise holds at {ent_last:.0f}%",
    loc="left", color=PALETTE["text"], fontsize=12,
)
ax.legend(loc="lower left", frameon=False, ncol=1, fontsize=9)

add_source_annotation(fig)
fig.tight_layout()
fig.savefig(
    FIGURES_DIR / "07_segment_nrr_trends.png",
    dpi=150, bbox_inches="tight",
)
plt.show()


Mid-market is the source of the headline decline. Its rolling NRR
falls roughly 24 percentage points across the analysis period —
from the mid-110s in early 2023 to the low 90s by the end of 2025 —
while Enterprise holds in the 110–120% band and SMB drifts from
~107% to ~99%. Rolling and raw lines move in step, which is the
intended diagnostic check: the smoothed signal is reliable, the
noise is real but small. The Enterprise raw line is visibly the
noisiest of the three (small denominator, ~40–50 retention
customers per quarter), so segment-level conclusions are stated
from the rolling average and not the raw series.

## Section 8 — Cohort triangle

Cohort = quarter of `first_paid_invoice_date`. Cohort starting ARR
is anchored at the end of the customer's **first full calendar
quarter after** billing began, which removes partial-quarter
dilution for customers who started part-way through a quarter.
Subsequent cells in each row show retained ARR as a percentage of
that cohort's starting ARR.

Cohorts with fewer than 5 customers are suppressed for signal
clarity. In the current dataset every billing-date cohort from
Q1 2021 to Q3 2025 clears that bar (smallest cohort size 10);
the Q4 2025 cohort is excluded because its anchor quarter (Q1
2026) sits past the snapshot date.

In [ ]:
cohort_in_book = customers[
    customers["first_paid_invoice_date"] <= SNAPSHOT_DATE
].copy()
cohort_in_book["cohort_q"] = cohort_in_book["first_paid_invoice_date"].dt.to_period("Q")
cohort_in_book["anchor_q"] = cohort_in_book["cohort_q"] + 1
cohort_in_book["anchor_qe"] = cohort_in_book["anchor_q"].dt.end_time.dt.normalize()

cohort_qe_index = pd.date_range(
    cohort_in_book["anchor_qe"].min(), SNAPSHOT_DATE, freq="QE",
)
cohort_in_book["anchor_pos"] = pd.Index(cohort_qe_index).get_indexer(
    cohort_in_book["anchor_qe"]
)
cohort_in_book = cohort_in_book[cohort_in_book["anchor_pos"] >= 0].copy()

cohort_active = subscriptions[
    subscriptions["month_end_date"].isin(cohort_qe_index)
    & (subscriptions["subscription_status"] == "Active")
]
cohort_qe_pivot = (
    cohort_active.pivot_table(
        index="customer_id",
        columns="month_end_date",
        values="arr_gbp",
        aggfunc="sum",
        fill_value=0.0,
    )
    .reindex(columns=cohort_qe_index, fill_value=0.0)
)

arr_aligned = (
    cohort_qe_pivot
    .reindex(cohort_in_book["customer_id"])
    .fillna(0.0)
    .to_numpy()
)
n_qe = len(cohort_qe_index)
anchor_positions = cohort_in_book["anchor_pos"].to_numpy()
quarters_since = np.arange(n_qe)[None, :]
target_pos = anchor_positions[:, None] + quarters_since
in_window = target_pos < n_qe
gathered = np.take_along_axis(
    arr_aligned, np.clip(target_pos, 0, n_qe - 1), axis=1,
)
gathered = np.where(in_window, gathered, 0.0)

cohort_label = cohort_in_book["cohort_q"].astype(str).to_numpy()
cohort_arr = (
    pd.DataFrame(gathered, index=cohort_label)
    .groupby(level=0)
    .sum()
    .sort_index()
)

cohort_anchor_pos = (
    cohort_in_book.groupby(cohort_in_book["cohort_q"].astype(str))["anchor_pos"]
    .first()
)
cohort_max_k = (n_qe - 1) - cohort_anchor_pos
visible_mask = (
    np.arange(n_qe)[None, :]
    <= cohort_max_k.reindex(cohort_arr.index).to_numpy()[:, None]
)
cohort_arr = cohort_arr.where(visible_mask).dropna(axis=1, how="all")
cohort_arr.columns = [f"q+{c}" for c in cohort_arr.columns]

cohort_sizes = (
    cohort_in_book.groupby(cohort_in_book["cohort_q"].astype(str))
    .size()
    .reindex(cohort_arr.index)
)
keep_cohorts = cohort_sizes[cohort_sizes >= 5].index
suppressed_cohorts = cohort_sizes[cohort_sizes < 5].index.tolist()

retention_pct = (
    cohort_arr.loc[keep_cohorts].divide(cohort_arr.loc[keep_cohorts].iloc[:, 0], axis=0)
)

retention_display = (retention_pct * 100).round(0)
retention_display.insert(0, "n", cohort_sizes.loc[keep_cohorts].astype(int).to_numpy())
retention_display


In [ ]:
from matplotlib.colors import LinearSegmentedColormap

cohort_cmap = LinearSegmentedColormap.from_list(
    "orbitstack_cohort",
    [PALETTE["accent"], PALETTE["background"], PALETTE["positive"]],
)

heatmap_data = retention_pct * 100
n_rows, n_cols = heatmap_data.shape
fig, ax = plt.subplots(figsize=(max(11, 0.6 * n_cols + 3), max(7, 0.42 * n_rows + 1)))

sns.heatmap(
    heatmap_data,
    ax=ax,
    cmap=cohort_cmap,
    center=100,
    vmin=70, vmax=170,
    annot=True, fmt=".0f",
    annot_kws={"size": 8, "color": PALETTE["text"]},
    cbar_kws={"label": "Retained ARR (% of cohort starting ARR)", "shrink": 0.7},
    linewidths=0.4, linecolor=PALETTE["background"],
    square=False,
    mask=heatmap_data.isna(),
)

ax.set_xlabel("Quarters since cohort start (q+0 = end of first full quarter post-billing)")
ax.set_ylabel("Cohort (quarter of first paid invoice)")
ax.set_title(
    "Pre-2024 cohorts expand past 120%; 2024+ cohorts plateau or decline by q+5",
    loc="left", color=PALETTE["text"], fontsize=12,
)
ax.tick_params(axis="x", rotation=0)
ax.tick_params(axis="y", rotation=0)

footer = (
    "n column shows cohort customer count. Cohorts with fewer than 5 customers "
    "are suppressed; none in this dataset trigger that threshold. The Q4 2025 "
    "cohort is excluded because its anchor quarter sits past the snapshot date."
)
fig.text(
    0.01, -0.04, footer,
    fontsize=8, color=PALETTE["neutral"], ha="left", va="top", wrap=True,
)
add_source_annotation(fig)

fig.tight_layout()
fig.savefig(
    FIGURES_DIR / "08_cohort_triangle.png",
    dpi=150, bbox_inches="tight",
)
plt.show()


Pre-2024 cohorts continue to expand past q+4: 2021 and 2022
billing-date cohorts typically reach 120–160% of starting ARR by
two to three years post-anchor, with the 2021Q2 cohort the
strongest at ~168% by q+17. Cohorts billed from Q2 2024 onwards
plateau or decline early — 2024Q2 sits at 91% by q+5 and 2024Q3
at 86% by q+4, both materially below the >110% trajectory
established by every pre-2024 cohort at the same tenure.

Combined with the segment cut in section 7, the cohort triangle
narrows the diagnostic: the expansion engine has stalled for
newer cohorts. The inflection clusters around the Q3 2024
FleetLogic emergence examined in section 9, and the absence of a
matching deterioration in pre-2024 cohorts argues against a
blanket macro explanation.

## Section 9 — Churn reason taxonomy

The section-7 segmentation cut places the NRR collapse in mid-market,
and the cohort triangle in section 8 ties the inflection to Q3 2024
onwards. The churn-reason log lets us test whether a specific
competitor entry coincides with that window.

Each churned customer carries a single primary reason captured by
Customer Success at the time of renewal loss. Reasons are grouped
into half-year buckets so that the FleetLogic emergence window is
visible without over-smoothing.

In [ ]:
churn_with_segment = churn_reasons.merge(
    customers[["customer_id", "segment"]], on="customer_id", how="left"
)
churn_with_segment["half_year"] = (
    churn_with_segment["churn_date"].dt.year.astype(str)
    + " H"
    + np.where(churn_with_segment["churn_date"].dt.month <= 6, "1", "2")
)
churn_with_segment["half_end"] = pd.to_datetime(
    np.where(
        churn_with_segment["churn_date"].dt.month <= 6,
        churn_with_segment["churn_date"].dt.year.astype(str) + "-06-30",
        churn_with_segment["churn_date"].dt.year.astype(str) + "-12-31",
    )
)

half_order = (
    churn_with_segment[["half_year", "half_end"]]
    .drop_duplicates()
    .sort_values("half_end")["half_year"]
    .tolist()
)

REASON_ORDER = [
    "Acquisition/closure",
    "Non-payment",
    "Other",
    "Product fit",
    "Cost",
    "Competitor - other",
    "Competitor - FleetLogic",
]

REASON_COLOURS = {
    "Acquisition/closure":     "#B7C3CF",
    "Non-payment":             "#8B3A2F",
    "Other":                   "#D1D5DB",
    "Product fit":             PALETTE["secondary"],
    "Cost":                    PALETTE["neutral"],
    "Competitor - other":      PALETTE["primary"],
    "Competitor - FleetLogic": PALETTE["accent"],
}

REASON_HATCHES = {
    "Acquisition/closure":     "",
    "Non-payment":             "...",
    "Other":                   "xxx",
    "Product fit":             "///",
    "Cost":                    "",
    "Competitor - other":      "",
    "Competitor - FleetLogic": "\\\\\\\\",
}

churn_by_half = (
    churn_with_segment.pivot_table(
        index="half_year",
        columns="primary_reason",
        values="customer_id",
        aggfunc="count",
        fill_value=0,
    )
    .reindex(index=half_order, columns=REASON_ORDER, fill_value=0)
)

post_mask = churn_with_segment["half_end"] >= pd.Timestamp("2024-07-01")
mid_post = churn_with_segment[
    post_mask & (churn_with_segment["segment"] == "Mid-market")
]
mid_post_fleetlogic = int(
    (mid_post["primary_reason"] == "Competitor - FleetLogic").sum()
)
mid_post_total = int(len(mid_post))
mid_post_share = (
    mid_post_fleetlogic / mid_post_total * 100 if mid_post_total else 0.0
)
fleetlogic_total = int(
    (churn_with_segment["primary_reason"] == "Competitor - FleetLogic").sum()
)
all_churn = int(len(churn_with_segment))

x_positions = np.arange(len(half_order))
bottoms = np.zeros(len(half_order))

fig, ax = plt.subplots(figsize=(11, 6))
for reason in REASON_ORDER:
    heights = churn_by_half[reason].to_numpy()
    ax.bar(
        x_positions, heights, bottom=bottoms,
        color=REASON_COLOURS[reason],
        edgecolor=PALETTE["background"], linewidth=1.0,
        hatch=REASON_HATCHES[reason],
        label=reason,
    )
    bottoms = bottoms + heights

fleetlogic_idx = half_order.index("2024 H2")
ax.axvline(
    fleetlogic_idx - 0.5,
    linestyle=":", color=PALETTE["accent"], linewidth=1.4,
)

max_bar_height = int(bottoms.max())

ax.annotate(
    "FleetLogic emerges (Q3 2024)",
    xy=(fleetlogic_idx - 0.5, max_bar_height * 0.92),
    xytext=(fleetlogic_idx - 1.9, max_bar_height * 1.05),
    color=PALETTE["accent"], fontsize=9,
    arrowprops=dict(arrowstyle="->", color=PALETTE["accent"], lw=1.0),
)

ax.set_xticks(x_positions)
ax.set_xticklabels(half_order, rotation=30, ha="right")
ax.set_xlabel("Half-year of churn")
ax.set_ylabel("Churned customers")
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.set_ylim(0, max_bar_height * 1.25)

ax.set_title(
    f"FleetLogic absent before H2 2024, then {mid_post_share:.0f}% of "
    f"mid-market post-emergence churn "
    f"({mid_post_fleetlogic} of {mid_post_total} exits)",
    loc="left", color=PALETTE["text"], fontsize=12,
)
ax.legend(
    loc="upper left", frameon=False, fontsize=9,
    ncol=2, title="Primary reason",
)

add_source_annotation(fig)
fig.tight_layout()
fig.savefig(
    FIGURES_DIR / "09_churn_reasons.png",
    dpi=150, bbox_inches="tight",
)
plt.show()

f"FleetLogic cited in {fleetlogic_total} of {all_churn} total exits; "f"{mid_post_fleetlogic} of {mid_post_total} mid-market exits post-emergence."


Churn volume rises from 2023 H2 onwards, but the mix reshapes more
sharply than the total. FleetLogic is absent from the taxonomy before
Q3 2024 and then becomes a named competitor in every half-year from
2024 H2. It accounts for a material share of mid-market churn in the
post-emergence window, while Enterprise customers do not cite it at
all. Cost and Product fit remain the dominant reasons for SMB exits
throughout.

The temporal alignment ties the segment pattern from section 7 — the
~24 percentage point fall in mid-market rolling NRR — to a specific
competitor-entry event rather than a generalised macro deterioration.
The finding is directional, not causal: a competitor-intelligence
workstream is flagged in section 12 to substantiate through win-loss
interviews and product comparisons.

## Section 10 — Expansion deep-dive

The NRR decomposition in section 6 attributed the headline decline to
a shortfall in expansion, not a lift in churn. This section looks at
seat trajectories directly for customers still active at the snapshot,
indexed to 100 at month 12 post-first-paid-invoice so cohorts are
comparable regardless of when they started billing.

Customers without at least 12 months of history are excluded from
eligibility. Acquisition era splits on `acquisition_date` (pre-2024
vs 2024+). Each panel shows the mean indexed-seats trajectory for the
segment × era group; group sizes are annotated in-panel.

In [ ]:
churned_ids = churn_reasons["customer_id"].unique()
still_active = customers[
    (~customers["customer_id"].isin(churned_ids))
    & (customers["first_paid_invoice_date"] <= SNAPSHOT_DATE)
].copy()
still_active["era"] = np.where(
    still_active["acquisition_date"] < pd.Timestamp("2024-01-01"),
    "Pre-2024", "2024+",
)

seat_panel = subscriptions[
    subscriptions["customer_id"].isin(still_active["customer_id"])
].merge(
    still_active[["customer_id", "first_paid_invoice_date", "segment", "era"]],
    on="customer_id", how="left",
)
seat_panel["months_since_start"] = (
    (seat_panel["month_end_date"].dt.year
     - seat_panel["first_paid_invoice_date"].dt.year) * 12
    + (seat_panel["month_end_date"].dt.month
       - seat_panel["first_paid_invoice_date"].dt.month)
)

seats_matrix = seat_panel.pivot_table(
    index="customer_id",
    columns="months_since_start",
    values="seats",
    aggfunc="first",
)
anchor_seats = seats_matrix[12]
eligible_mask = anchor_seats.notna()
indexed_seats = (
    seats_matrix.loc[eligible_mask]
    .divide(anchor_seats.loc[eligible_mask], axis=0)
    * 100
)

customer_meta = still_active.set_index("customer_id")[["segment", "era"]]
indexed_with_meta = indexed_seats.join(customer_meta, how="inner")

mean_by_group = (
    indexed_with_meta.groupby(["segment", "era"], observed=False)
    .mean(numeric_only=True)
)
counts_by_group = (
    indexed_with_meta.groupby(["segment", "era"], observed=False).size()
)

ERA_ORDER = ["Pre-2024", "2024+"]
X_MAX = 36

fig, axes = plt.subplots(
    nrows=2, ncols=3, figsize=(13, 7.5),
    sharey=True, sharex=True,
)

for row_i, era in enumerate(ERA_ORDER):
    for col_i, seg in enumerate(SEGMENT_ORDER):
        ax = axes[row_i, col_i]
        key = (seg, era)
        if key in mean_by_group.index and not mean_by_group.loc[key].dropna().empty:
            trajectory = mean_by_group.loc[key]
            trajectory = trajectory[
                (trajectory.index >= 0) & (trajectory.index <= X_MAX)
            ].dropna()
            ax.plot(
                trajectory.index.to_numpy(), trajectory.to_numpy(),
                color=SEGMENT_COLOURS[seg],
                linestyle=SEGMENT_LINESTYLES[seg],
                linewidth=2.3, marker="o", markersize=3.5,
                markeredgewidth=0,
            )
            n_customers = int(counts_by_group.loc[key])
            ax.text(
                0.03, 0.96, f"n = {n_customers}",
                transform=ax.transAxes,
                fontsize=9, color=PALETTE["neutral"],
                va="top", ha="left",
            )
        ax.axhline(100, linestyle=":", color=PALETTE["neutral"], linewidth=1.0)
        ax.axvline(18, linestyle=":", color=PALETTE["accent"], linewidth=1.1)

        if row_i == 0:
            ax.set_title(seg, loc="left", fontsize=11, color=PALETTE["text"])
        if col_i == 0:
            ax.set_ylabel(
                f"{era}\nIndexed seats (month 12 = 100)",
                fontsize=10,
            )
        if row_i == len(ERA_ORDER) - 1:
            ax.set_xlabel("Months since first paid invoice")

axes[0, 0].set_xlim(0, X_MAX)

mid_pre_24 = mean_by_group.loc[("Mid-market", "Pre-2024")]
mid_post_24 = mean_by_group.loc[("Mid-market", "2024+")]
ent_pre_24 = mean_by_group.loc[("Enterprise", "Pre-2024")]
mid_pre_24_m24 = float(mid_pre_24.loc[24])
mid_post_24_last = float(mid_post_24.dropna().iloc[-1])
ent_pre_24_m24 = float(ent_pre_24.loc[24])

fig.suptitle(
    "Mid-market 2024+ cohorts flatline at month-12 seats "
    f"({mid_post_24_last:.0f}% at latest tenure); pre-2024 Mid-market "
    f"grows to {mid_pre_24_m24:.0f}% by month 24, Enterprise to {ent_pre_24_m24:.0f}%",
    x=0.01, y=0.995, ha="left", fontsize=12, color=PALETTE["text"],
)

axes[0, 1].text(
    18.4, 97.5, "month 18",
    color=PALETTE["accent"], fontsize=8, va="top", ha="left",
)

add_source_annotation(fig)
fig.tight_layout(rect=(0, 0, 1, 0.96))
fig.savefig(
    FIGURES_DIR / "10_expansion_trajectory.png",
    dpi=150, bbox_inches="tight",
)
plt.show()


The mechanical cause of the expansion collapse is visible directly
in seat data. Pre-2024 Mid-market and Enterprise cohorts compound
seats steadily past the month-12 anchor, reaching roughly 112% and
121% of anchor seats by month 24 respectively. 2024+ cohorts do
something qualitatively different: Mid-market plateaus within the
first 12 months of billing and stays broadly at the anchor level,
while 2024+ Enterprise and SMB sit close to 100% with small samples.
SMB pre-2024 is flat throughout — consistent with the segment having
modest expansion at best.

The flatline for 2024+ Mid-market sits on top of the FleetLogic
emergence in section 9: newer customers either do not buy additional
seats or actively contract in the first year post-billing. Because
expansion ARR is the numerator movement that lifts NRR above GRR,
losing expansion on new cohorts pulls the denominator-weighted
portfolio NRR down exactly as observed in section 5. The vertical
dotted line at month 18 marks the tenure point from which 2023-H1
acquisitions would first encounter the mid-2024 demand shift.